In [ ]:
import pyspark.sql.functions as F

# Datamart Gold: Customer Monthly Spending

Este notebook calcula el gasto mensual y la cantidad de transacciones por usuario a lo largo del tiempo, permitiendo analizar la evolucion del valor del cliente mes a mes en herramientas de BI.

In [ ]:
# 1. Lectura de tablas Silver
df_trans = spark.table('castor.silver.slv_fact_transactions')
df_users = spark.table('castor.silver.slv_dim_users')

In [ ]:
# 2. Agrupacion de transacciones por usuario y ano-mes
df_monthly_spend = df_trans.withColumn(
    'ano_mes', F.date_format('fecha_transaccion', 'yyyy-MM')
).groupBy('id_usuario', 'ano_mes').agg(
    F.round(F.sum('monto'), 2).alias('gasto_mensual'),
    F.count('id_transaccion').alias('transacciones_del_mes')
)

In [ ]:
# 3. Enriquecimiento con datos basicos del usuario (Nombre)
df_result = df_monthly_spend.join(
    df_users.select('id_usuario', 'nombre'),
    on='id_usuario',
    how='left'
).select(
    'id_usuario',
    'nombre',
    'ano_mes',
    'gasto_mensual',
    'transacciones_del_mes'
).orderBy('id_usuario', 'ano_mes')

display(df_result.limit(10))

In [ ]:
# 4. Guardar en la capa Gold
df_result.write.mode('overwrite').saveAsTable('castor.gold.gld_customer_monthly_spending')
print('Tabla castor.gold.gld_customer_monthly_spending guardada exitosamente.')